[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab3_Agentic_RAG_LangGraph_Student.ipynb)


# Lab 3: Agentic RAG with LangGraph
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Basic RAG retrieves once and answers. But complex clinical questions need more — query rewriting, self-correction, multi-step reasoning. You'll build an agentic RAG system that thinks before retrieving and validates its own answers.

### Objective
- Build a LangGraph agentic RAG (uses Session 5 skills)
- Add query rewriting before retrieval
- Add document grading (self-RAG)
- Add corrective re-retrieval
- Compare with basic RAG from Lab 1

### Design choices (agentic RAG)
- **Query rewrite**: maps vague user language to retrieval-friendly clinical terms; can retry with a different angle.
- **Grade documents**: cheap self-check — are these chunks actually about the question? If not, rewrite and retrieve again.
- **Max iterations**: caps cost/latency; then we still **generate** with best-effort context.

**Chunking / embeddings / vector DB** use Lab 1 defaults here so you mostly isolate the effect of the **agent workflow**.


---
## Setup

Install packages, add **`OPENAI_API_KEY`** to Colab **Secrets**. Upload **`chroma_wt.zip`** (from Lab 1) when prompted in Cell 1. Restart runtime if imports fail after install.

In [ ]:
!pip install -q llama-parse langchain langchain-text-splitters langchain-openai langchain-community langchain-chroma chromadb langgraph deepeval

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# os.environ['LLAMA_CLOUD_API_KEY'] = userdata.get('LLAMA_CLOUD_API_KEY')  # only needed if rebuilding from scratch
# from google.colab import files; files.upload(); PDF_PATH = '/content/WT.pdf'  # only needed if rebuilding

---
## Cell 1 — Load retriever from Lab 1

Load the **persisted Chroma vector store** built in Lab 1. No re-parsing or re-embedding needed. The rebuild option is commented out for reference (or if running this lab standalone).

In [ ]:
from google.colab import files
import zipfile

files.upload()  # pick chroma_wt.zip (from Lab 1)
with zipfile.ZipFile('chroma_wt.zip', 'r') as z:
    z.extractall('.')
print("chroma_wt extracted → ./chroma_wt")

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# Load persisted vector store from Lab 1 — no re-parsing or re-embedding needed
vs = Chroma(
    collection_name='wt_good',
    embedding_function=OpenAIEmbeddings(model='text-embedding-3-small'),
    persist_directory='./chroma_wt',
)
retriever = vs.as_retriever(search_kwargs={'k': 4})
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
print(f"Retriever ready ({vs._collection.count()} chunks, loaded from Lab 1)")

# ── optional: rebuild from scratch (uncomment if ./chroma_wt doesn't exist) ──────────────────────
# import os
# from llama_parse import LlamaParse
# from langchain_core.documents import Document
# from langchain_text_splitters import RecursiveCharacterTextSplitter
# raw = LlamaParse(api_key=os.environ['LLAMA_CLOUD_API_KEY'], result_type='markdown').load_data(PDF_PATH)
# lc_docs = [Document(page_content=d.text, metadata={'source': 'WT.pdf'}) for d in raw]
# chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(lc_docs)
# vs = Chroma.from_documents(chunks, OpenAIEmbeddings(model='text-embedding-3-small'), collection_name='wt_good', persist_directory='./chroma_wt')
# ─────────────────────────────────────────────────────────────────────────────────────────────────

## Cell 2 — Define graph state

In [ ]:
from typing import TypedDict, List

class AgenticRAGState(TypedDict):
    question: str
    rewritten_question: str
    documents: List[str]
    grade: str  # 'relevant' or 'irrelevant'
    answer: str
    iteration: int

## Cell 3 — Define nodes

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# TODO: rewrite_query_node — use LLM to rewrite question for clearer retrieval.
def rewrite_query_node(state: AgenticRAGState) -> dict:
    # Hint: prompt LLM to refine the original question with clinical terminology
    ...
    return {'rewritten_question': ..., 'iteration': state.get('iteration', 0) + 1}

# TODO: retrieve_node
def retrieve_node(state: AgenticRAGState) -> dict:
    q = state['rewritten_question']
    # Hint: docs = retriever.invoke(q)
    docs = ...
    return {'documents': [d.page_content for d in docs]}

# TODO: grade_documents_node — LLM judges if docs answer the question
def grade_documents_node(state: AgenticRAGState) -> dict:
    # Hint: ask LLM 'relevant' or 'irrelevant'
    grade = ...
    return {'grade': grade}

# TODO: generate_answer_node
def generate_answer_node(state: AgenticRAGState) -> dict:
    ctx = '\n\n'.join(state['documents'])
    # Hint: prompt LLM with context + question
    ans = ...
    return {'answer': ans}

## Cell 4 — Conditional edges

In [ ]:
MAX_ITER = 2

def route_after_grade(state: AgenticRAGState) -> str:
    # TODO: if relevant → 'generate'
    # if irrelevant and iteration < MAX_ITER → 'rewrite'
    # else → 'generate' (best effort)
    ...

## Cell 5 — Build & compile graph

In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(AgenticRAGState)
# TODO: add nodes (rewrite, retrieve, grade, generate)
# TODO: edges: START → rewrite → retrieve → grade → (conditional) → generate → END
...
agentic_app = g.compile()

## Cell 6 — Test on hard clinical questions

In [ ]:
questions = [
    'Show me Wilms tumor staging that requires both chemo and radiation',
    'What are the late effects of treatment in Stage I patients?',
    'Compare treatment regimens for favorable vs unfavorable histology',
]
for q in questions:
    out = agentic_app.invoke({'question': q, 'iteration': 0})
    print('Q:', q)
    print('A:', out['answer'])
    print('iterations:', out['iteration'])
    print('-' * 60)

## Cell 7 — Compare with basic RAG

In [ ]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

basic_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer ONLY from the context.\n\n{context}'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def _basic_rag(d):
    docs = retriever.invoke(d['input'])
    answer = (basic_prompt | llm | StrOutputParser()).invoke({'context': format_docs(docs), 'input': d['input']})
    return {'answer': answer, 'context': docs}

basic_chain = RunnableLambda(_basic_rag)

for q in questions:
    basic = basic_chain.invoke({'input': q})['answer']
    agentic = agentic_app.invoke({'question': q, 'iteration': 0})['answer']
    print(f'\nQ: {q}\n--BASIC--\n{basic}\n--AGENTIC--\n{agentic}\n')

## Cell 8 — DeepEval comparison

In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase

metrics = [FaithfulnessMetric(threshold=0.7, model='gpt-4o-mini'),
           AnswerRelevancyMetric(threshold=0.7, model='gpt-4o-mini')]

# TODO: for each question run both pipelines, build LLMTestCase, score, compare
...

## Stretch — Add a `needs_more_info` node
If the LLM marks the question as ambiguous, ask the user for clarification before retrieval.

## KHCC Connection
Agentic self-correcting RAG is what closes the safety gap for clinical assistants — when the system can recognize 'I don't have a good answer here' it can re-query or escalate, instead of hallucinating.